In [7]:
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin


class MyKNNClassifier(BaseEstimator, ClassifierMixin):


    def __init__(self, n_neighbors=3, metric="euclidean", weighted=False):
        self.n_neighbors = int(n_neighbors)
        self.metric = metric
        self.weighted = bool(weighted)


        self.X_train = None
        self.y_train = None
        self.labels = None
        self.trainIndex = None

    def fit(self, X, Y):

        self.X_train = np.asarray(X)
        self.y_train = np.asarray(Y)

        self.labels, self.trainIndex = np.unique(Y, return_inverse=True)

        return self

    def predict(self, X):

        X = np.asarray(X)
        if self.metric == "euclidean":

            distanceMatrix = np.sqrt(np.maximum(
                np.sum(X * X, axis=1, keepdims=True) +
                np.sum(self.X_train * self.X_train, axis=1, keepdims=True).T -
                2.0 * (X @ self.X_train.T)
                , 0.0)
            , dtype=X.dtype)
        else:  # raveshe manhattan

            distanceMatrix = np.sum(np.abs(X[:, None, :] - self.X_train[None, :, :]), axis=2)

        #shakhes haye k neighbor nazdik
        kNeighborsIndices = np.argpartition(distanceMatrix, kth=self.n_neighbors - 1, axis=1)[:, :self.n_neighbors]

        #their distance
        kNeighborsDistances = np.take_along_axis(distanceMatrix, kNeighborsIndices, axis=1)

        if self.weighted:

            weights = 1.0 / (kNeighborsDistances + 1e-12)
        else:
            weights = np.ones_like(kNeighborsDistances, dtype=float)

        #rows -> data
        #column -> class
        candidateLabels = np.zeros((X.shape[0], self.labels.shape[0]), dtype=float)

        np.add.at(candidateLabels, (np.arange(X.shape[0])[:, None], self.trainIndex[kNeighborsIndices]), weights)


        predictedLabelIndex = np.argmax(candidateLabels, axis=1)
        return self.labels[predictedLabelIndex]


In [8]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

irisData = load_iris()
X = irisData.data
#numerical labels
y = irisData.target
targets = irisData.target_names

#scaler ->normalizing
pipeLine = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", MyKNNClassifier())
])

crossValidation = StratifiedKFold()

parameters = {
    "knn__n_neighbors": [1, 2, 3, 4, 5, 10],
    "knn__metric": ["euclidean", "manhattan"],
    "knn__weighted": [False, True],
}

# Accuracy = (TP + TN) / (TP + TN + FP + FN)
# Precision = TP / (TP + FP)
# Recall = TP / (TP + FN)
# F1 = 2 * (Precision * Recall) / (Precision + Recall)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision_macro",
    "recall": "recall_macro",
    "f1": "f1_macro",
}

grid = GridSearchCV(
    estimator=pipeLine,
    param_grid=parameters,
    cv=crossValidation,
    scoring=scoring,
    #choose best model according to f1
    refit="f1",
    n_jobs=-1,
    return_train_score=False
)

grid.fit(X, y)

bestPipeline = grid.best_estimator_
predictionCrossValidation = cross_val_predict(bestPipeline, X, y, cv=crossValidation)

classificationReport = classification_report(y, predictionCrossValidation, target_names=targets, digits=3)

metrics = {
    "accuracy": accuracy_score(y, predictionCrossValidation),
    "precision_macro": precision_score(y, predictionCrossValidation, average="macro"),
    "recall_macro": recall_score(y, predictionCrossValidation, average="macro"),
    "f1_macro": f1_score(y, predictionCrossValidation, average="macro"),
}

print("Best params res:")
print(grid.best_params_)
print("report for Cross-validated classification:")
print(classificationReport)
#miangin
print("aggr scores:")
print({k: round(v, 4) for k, v in metrics.items()})

Best params res:
{'knn__metric': 'euclidean', 'knn__n_neighbors': 10, 'knn__weighted': True}
report for Cross-validated classification:
              precision    recall  f1-score   support

      setosa      1.000     1.000     1.000        50
  versicolor      0.941     0.960     0.950        50
   virginica      0.959     0.940     0.949        50

    accuracy                          0.967       150
   macro avg      0.967     0.967     0.967       150
weighted avg      0.967     0.967     0.967       150

aggr scores:
{'accuracy': 0.9667, 'precision_macro': 0.9668, 'recall_macro': 0.9667, 'f1_macro': 0.9667}
